# 🧪 Aula 16 — Prática: Data Version Control (DVC)

**Grupo de Estudos em MLOps — CEIA/UFG**

Nesta prática você vai versionar **dados, modelos e pipelines** com o DVC, usando o mesmo modelo mental que você já conhece do Git:

| Parte | Tema | Comandos-chave |
|---|---|---|
| 1 | Inicializando o projeto | `git init`, `dvc init` |
| 2 | Versionando dados + viagem no tempo | `dvc add`, `git tag`, `dvc checkout` |
| 3 | Remote: o "GitHub dos dados" | `dvc remote`, `dvc push`, `dvc pull` |
| 4 | Pipeline reprodutível | `dvc.yaml`, `dvc repro`, `dvc dag` |
| 5 | Métricas, params e experimentos | `dvc metrics/params diff`, `dvc exp` |

> ✅ **Não precisa de GPU** — roda em qualquer runtime do Colab ou localmente.
>
> 🐳 **Prefere rodar fora do Colab?** A pasta `atividade/` traz o mesmo pipeline como código-fonte (`src/`) e um `docker-compose` que sobe um **remote S3 de verdade (MinIO)** — veja o `README.md` da atividade. O notebook usa um remote em diretório local para ser auto-contido.

In [ ]:
# Instala as dependências (git já vem instalado no Colab).
%pip install -q "dvc>=3.0" scikit-learn pandas pyyaml

## Parte 1 — Um projeto git + dvc do zero

O DVC **não substitui o Git — ele trabalha sobre um repositório Git**. A divisão de responsabilidades é o conceito central da aula:

| O que | Onde fica | Quem versiona |
|---|---|---|
| Código, configs, params | Repositório Git | Git |
| **Metadados** dos dados (`.dvc`, `dvc.lock`) | Repositório Git | Git |
| **Conteúdo** dos dados/modelos | Cache local + remote | DVC |

Vamos criar um workspace limpo (fora de qualquer repositório existente) e inicializar os dois.

In [ ]:
import os
import shutil
import stat
import sys

IN_COLAB = "google.colab" in sys.modules
BASE = "/content/dvc-lab" if IN_COLAB else os.path.join(os.path.expanduser("~"), "dvc-lab")
REMOTE_DIR = os.path.join(os.path.dirname(BASE), "dvc-remote")


def force_rmtree(path):
    """rmtree que lida com arquivos read-only do .git (necessário no Windows)."""
    def onerror(func, p, _exc):
        os.chmod(p, stat.S_IWRITE)
        func(p)
    shutil.rmtree(path, onerror=onerror)


# Recomeça do zero a cada execução completa do notebook.
for path in (BASE, REMOTE_DIR):
    if os.path.exists(path):
        force_rmtree(path)
os.makedirs(os.path.join(BASE, "src"))

%cd {BASE}

!git init -q
!git branch -M main
!git config user.name "Estudante MLOps"
!git config user.email "estudante@ceia.ufg.br"

!dvc init -q
!git commit -q -m "chore: inicializa git + dvc"
!git log --oneline

In [ ]:
# O `dvc init` criou o diretório .dvc/ (config + futuro cache) e o .dvcignore.
# Repare: quase nada ainda — o DVC é leve até você começar a rastrear dados.
!ls -a
print()
!ls .dvc

## Parte 2 — Versionando o primeiro dado

Vamos gerar um dataset "bruto". Na vida real ele viria de um banco, uma API, sensores — aqui, um script o gera sinteticamente.

**Regra de bolso que vale para o curso inteiro:**

- Dado **FONTE** (chega de fora, imutável) → `dvc add`
- Dado **DERIVADO** (produzido a partir da fonte) → `outs` de um estágio do pipeline (Parte 4)

O script abaixo é cópia fiel de `atividade/src/get_data.py`.

In [ ]:
%%writefile src/get_data.py
"""Gera o dataset 'bruto' da prática (dado FONTE -> versionado com dvc add).

Re-execute com outros argumentos para simular a chegada de dados novos:
    python src/get_data.py                  # v1: 2000 amostras
    python src/get_data.py --samples 4000   # v2: 'chegou mais dado'
"""
import argparse
from pathlib import Path

import pandas as pd
from sklearn.datasets import make_classification

parser = argparse.ArgumentParser()
parser.add_argument("--samples", type=int, default=2000)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--output", type=Path, default=Path("data/raw/data.csv"))
args = parser.parse_args()

features, target = make_classification(
    n_samples=args.samples,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    flip_y=0.05,
    random_state=args.seed,
)
df = pd.DataFrame(features, columns=[f"feature_{i}" for i in range(features.shape[1])])
df["target"] = target

args.output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(args.output, index=False)
print(f"Dataset gerado: {args.output} ({len(df)} linhas, {args.output.stat().st_size / 1024:.1f} KB)")

In [ ]:
!python src/get_data.py --samples 2000
print()
!dvc add data/raw/data.csv

print("\n--- data/raw/data.csv.dvc (o PONTEIRO que vai para o Git) ---")
!cat data/raw/data.csv.dvc

print("--- data/raw/.gitignore (o DVC tirou o dado real do caminho do Git) ---")
!cat data/raw/.gitignore

In [ ]:
# Onde o conteúdo foi parar? No CACHE, indexado pelo HASH do conteúdo
# (compare o caminho do arquivo abaixo com o md5 do .dvc acima).
!find .dvc/cache -type f

### 🔎 O que acabou de acontecer

O `dvc add`:

1. Calculou o **MD5** do arquivo;
2. Moveu o conteúdo para o **cache content-addressable** (`.dvc/cache/files/md5/<2 primeiros dígitos>/<resto>`) e deixou um *link* no workspace;
3. Criou o ponteiro `data.csv.dvc` (5 linhas de YAML!) e um `.gitignore` para o dado real.

Indexar **pelo conteúdo, e não pelo nome**, é a mesma ideia dos objetos do Git — e dá deduplicação e checkout instantâneo de graça. Agora o snapshot: commitamos o **ponteiro** (não o dado) e marcamos com uma tag.

In [ ]:
!git add src/get_data.py data/raw/data.csv.dvc data/raw/.gitignore
!git commit -q -m "feat: dataset v1 (2000 amostras)"
!git tag v1.0
!git log --oneline

## Parte 3 — Remote: o "GitHub dos dados"

O cache é local. Para o time inteiro (e o CI) acessar os dados, o DVC usa um **remote**: S3, GCS, Azure, SSH, MinIO... e também um simples diretório — que usaremos aqui para o notebook ser auto-contido.

O fluxo de equipe espelha o do Git:

```text
colega A: git push  +  dvc push
colega B: git pull  +  dvc pull   →  mesmo código, MESMOS dados
```

> 🐳 No caminho **Docker** da atividade, este mesmo passo usa um **MinIO (S3 self-hosted)** — com console web onde você VÊ os objetos chegando a cada `dvc push`.

In [ ]:
!dvc remote add -d localremote {REMOTE_DIR}
!git add .dvc/config
!git commit -q -m "chore: configura remote de dados"

!dvc push
print("\n--- objetos no remote (mesma estrutura content-addressable do cache) ---")
!find {REMOTE_DIR} -type f

In [ ]:
# "Chegou mais dado": geramos a v2 do dataset (4000 amostras) e versionamos.
!python src/get_data.py --samples 4000
!dvc add data/raw/data.csv
!git add data/raw/data.csv.dvc
!git commit -q -m "feat: dataset v2 (4000 amostras)"
!git tag v2.0
!dvc push

print("\n--- o 'diff' de dados entre v1 e v2 é o diff dos PONTEIROS ---")
!git diff v1.0 v2.0 -- data/raw/data.csv.dvc

In [ ]:
# ⏰ VIAGEM NO TEMPO — o momento uau da aula:
#   git checkout  -> restaura os PONTEIROS da época
#   dvc checkout  -> materializa os DADOS que eles apontam
import pandas as pd


def n_rows():
    return len(pd.read_csv("data/raw/data.csv"))


print(f"agora (v2)           : {n_rows()} linhas")

!git checkout -q v1.0 && dvc checkout -q
print(f"após checkout da v1.0: {n_rows()} linhas")

!git checkout -q main && dvc checkout -q
print(f"de volta à main (v2) : {n_rows()} linhas")

### 🔎 Interpretação

- O `git diff` entre versões de dados é **legível** (mudou o md5 e o tamanho) mesmo que o dado tenha gigabytes.
- O `dvc checkout` foi **instantâneo**: nada foi re-baixado nem re-copiado — só os links para o cache foram refeitos.
- O cache guarda as duas versões **sem duplicar** o que não mudou (aqui os arquivos são diferentes; num diretório com 10.000 imagens onde 10 mudaram, só as 10 novas ocupariam espaço).

**Pergunta**: e se um colega clonasse este repositório agora? Ele teria os `.dvc`, mas não os dados — bastaria um `dvc pull`. É exatamente o fluxo do caminho Docker da atividade.

## Parte 4 — Pipeline reprodutível: `dvc.yaml` + `dvc repro`

Um experimento raramente é um script só: é **preparar → treinar → avaliar**. Vamos declarar esse DAG no `dvc.yaml`, com cada estágio dizendo:

- `cmd`: o que executar
- `deps`: de que arquivos depende (código **e** dados)
- `params`: que chaves do `params.yaml` o afetam
- `outs`: o que produz (vai para o cache/remote como qualquer dado)

O `dvc repro` compara **hashes** com o `dvc.lock` e re-executa **só o que mudou** — um "Makefile ciente de conteúdo".

Os três scripts abaixo são cópias fiéis de `atividade/src/` (lá com documentação completa).

In [ ]:
%%writefile src/prepare.py
"""Estágio prepare: divide o dado bruto em treino/teste (params: prepare.*)."""
from pathlib import Path

import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

params = yaml.safe_load(Path("params.yaml").read_text())

df = pd.read_csv("data/raw/data.csv")
train_df, test_df = train_test_split(
    df,
    test_size=params["prepare"]["test_size"],
    random_state=params["base"]["seed"],
    stratify=df["target"],
)

out = Path("data/prepared")
out.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out / "train.csv", index=False)
test_df.to_csv(out / "test.csv", index=False)
print(f"prepare: {len(train_df)} treino / {len(test_df)} teste")

In [ ]:
%%writefile src/train.py
"""Estágio train: treina e serializa o modelo (params: train.*)."""
from pathlib import Path

import joblib
import pandas as pd
import yaml
from sklearn.ensemble import RandomForestClassifier

params = yaml.safe_load(Path("params.yaml").read_text())

df = pd.read_csv("data/prepared/train.csv")
model = RandomForestClassifier(
    n_estimators=params["train"]["n_estimators"],
    max_depth=params["train"]["max_depth"],
    random_state=params["base"]["seed"],
    n_jobs=-1,
)
model.fit(df.drop(columns=["target"]), df["target"])

Path("models").mkdir(exist_ok=True)
joblib.dump(model, "models/model.pkl")
print(f"train: RandomForest(n_estimators={params['train']['n_estimators']}, "
      f"max_depth={params['train']['max_depth']})")

In [ ]:
%%writefile src/evaluate.py
"""Estágio evaluate: métricas em eval/metrics.json + pares para plots."""
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

df = pd.read_csv("data/prepared/test.csv")
model = joblib.load("models/model.pkl")
predictions = model.predict(df.drop(columns=["target"]))

metrics = {
    "accuracy": round(float(accuracy_score(df["target"], predictions)), 4),
    "precision": round(float(precision_score(df["target"], predictions)), 4),
    "recall": round(float(recall_score(df["target"], predictions)), 4),
    "f1": round(float(f1_score(df["target"], predictions)), 4),
}

Path("eval").mkdir(exist_ok=True)
Path("eval/metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
pd.DataFrame({"actual": df["target"], "predicted": predictions}).to_csv(
    "eval/predictions.csv", index=False
)
print(f"evaluate: {json.dumps(metrics)}")

In [ ]:
%%writefile params.yaml
# Hiperparâmetros do pipeline — fora do código para o DVC saber
# exatamente quais estágios cada mudança invalida.
base:
  seed: 42

prepare:
  test_size: 0.25

train:
  n_estimators: 100
  max_depth: 5

In [ ]:
%%writefile dvc.yaml
# Pipeline: prepare -> train -> evaluate
# data/raw/data.csv é DEP (dado fonte, versionado com dvc add);
# tudo que os estágios produzem é OUT (dado derivado, regenerável).
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
      - data/raw/data.csv
    params:
      - base.seed
      - prepare.test_size
    outs:
      - data/prepared

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/prepared
    params:
      - base.seed
      - train.n_estimators
      - train.max_depth
    outs:
      - models/model.pkl

  evaluate:
    cmd: python src/evaluate.py
    deps:
      - src/evaluate.py
      - models/model.pkl
      - data/prepared
    metrics:
      - eval/metrics.json:
          cache: false
    plots:
      - eval/predictions.csv:
          template: confusion
          x: actual
          y: predicted
          cache: false

In [ ]:
# Primeira execução: o DVC roda o DAG inteiro e registra os hashes no dvc.lock.
!dvc repro
print()
!git add . && git commit -q -m "feat: pipeline prepare->train->evaluate"
!dvc push -q

In [ ]:
# O DAG que o DVC montou a partir de deps/outs:
!dvc dag

# E a prova da memoização: rodar de novo SEM mudar nada...
print("\n--- dvc repro sem nenhuma mudança ---")
!dvc repro

### 🔎 Interpretação

- Na segunda execução, **nada rodou** ("didn't change, skipping"): os hashes de todas as deps batem com o `dvc.lock`.
- O `dvc.lock` (versionado no Git) registra os hashes exatos de cada dep/out — é o **certificado de reprodutibilidade** do pipeline: qualquer pessoa com este commit + `dvc pull` + `dvc repro` chega ao mesmo resultado.
- `models/model.pkl` é um `outs` como outro qualquer: **o modelo está versionado** e viaja no tempo junto com dado e código.

## Parte 5 — Métricas, params e experimentos

Agora a recompensa por ter declarado `params` e `metrics`: comparar execuções vira uma operação de **diff**, igual código.

In [ ]:
!dvc metrics show

In [ ]:
# Mudamos UM hiperparâmetro de treino...
import pathlib

import yaml

params = yaml.safe_load(pathlib.Path("params.yaml").read_text())
params["train"]["n_estimators"] = 300
pathlib.Path("params.yaml").write_text(yaml.safe_dump(params, sort_keys=False))

# ...e o dvc repro re-executa SÓ train e evaluate — prepare é pulado,
# porque nenhuma dependência dele mudou. Confira na saída:
!dvc repro

In [ ]:
# O que mudou em relação ao último commit? Params e métricas, lado a lado:
!dvc params diff
print()
!dvc metrics diff

In [ ]:
# Experimentos: variações baratas SEM poluir o histórico do Git.
# Cada `dvc exp run` roda o pipeline com um override e guarda o resultado
# como referência escondida — compare tudo com `dvc exp show`.
!git add . && git commit -q -m "exp: n_estimators=300"

!dvc exp run --set-param train.max_depth=10
print()
!dvc exp run --set-param train.max_depth=2
print()
!dvc exp show --md

### 🔎 Interpretação

- Cada linha do `dvc exp show` é uma execução completa: **params → métricas**, sem nenhum commit extra no histórico.
- Gostou de um experimento? `dvc exp apply <nome>` traz os arquivos dele para o workspace, e aí sim você commita o vencedor. Descartou? `dvc exp remove`.
- A matriz de confusão declarada em `plots` pode ser renderizada com `dvc plots show` (gera um HTML em `dvc_plots/`) — experimente localmente.
- Em equipe, o **DVC Studio** oferece essa tabela num painel web colaborativo — papel análogo ao MLflow Tracking, que já vimos: os dois se **complementam** (dados/pipeline vs tracking/registry).

## 🏁 Conclusão e reflexão

Você fechou o ciclo completo do DVC:

| Etapa | O que fizemos | Comando-chave |
|---|---|---|
| Versionar dados | Ponteiros `.dvc` no Git, conteúdo no cache | `dvc add` |
| Viajar no tempo | Restaurar dado + código de qualquer tag | `git checkout` + `dvc checkout` |
| Compartilhar | Remote content-addressable | `dvc push` / `dvc pull` |
| Reproduzir | DAG com memoização por hash | `dvc repro` + `dvc.lock` |
| Comparar | Params e métricas como diffs | `dvc metrics/params diff`, `dvc exp` |

**Para discutir no encontro:**

1. Por que indexar o cache **pelo hash do conteúdo** resolve deduplicação e viagem no tempo ao mesmo tempo?
2. O que aconteceria se você editasse `data/raw/data.csv` na mão, sem `dvc add`? Como o DVC perceberia?
3. No `dvc repro` da Parte 5, por que o estágio `prepare` foi pulado? E se você mudasse `base.seed`?
4. `models/model.pkl` versionado como `outs` substitui um *model registry* (MLflow)? O que cada abordagem dá e não dá?
5. Quando o DVC **não** é a ferramenta certa? (Pense: tabelas gigantes em lakehouse, streaming, features online.)

🐳 **Próximo passo**: rode o caminho **Docker** da atividade (`atividade/README.md`) — lá o remote é um **MinIO (S3 de verdade)** com console web, e você simula o fluxo de equipe completo: um "colega" clonando o repositório e recuperando os dados com `dvc pull`.